# VAE para CelebA

In [1]:
!pip install -q numpy pandas Pillow matplotlib tqdm torch torchvision kaggle

import os
import json
import random

import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando {DEVICE}.')

## Datos de entrenamiento

In [3]:
# Credenciales de Kaggle:
USERNAME = 'fernandoftis'
KEY = '5327d7ee02f8e41a6b1a131c44c0325f'

# Escribir credenciales de Kaggle:
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json = os.path.join(kaggle_dir, 'kaggle.json')
with open(kaggle_json, 'w') as f:
    json.dump({'username': USERNAME, 'key': KEY}, f)
os.chmod(kaggle_json, 0o600)

# Descargar y descomprimir el dataset CelebA:
#!kaggle datasets download -d jessicali9530/celeba-dataset --unzip -p celeba

In [4]:
class CelebA(Dataset):

    def __init__(self, img_dir='celeba/img_align_celeba/img_align_celeba/', attr_file='celeba/list_attr_celeba.csv', img_size=128):
        self.img_dir = img_dir
        self.img_size = img_size

        raw = np.genfromtxt(attr_file, delimiter=',', dtype=str)  # [N+1, 41]
        self.attribute_names = raw[0, 1:].tolist()  # [40].
        self.image_files = raw[1:, 0].tolist()  # [N].
        self.attrs = ((raw[1:, 1:].astype(np.int8) + 1) // 2).astype(np.int8)  # [N, 40]

        self.transform = transforms.Compose([
            transforms.CenterCrop(178),   # 178x218 -> 178x178
            transforms.Resize(img_size),  # 178x178 -> img_size x img_size
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(os.path.join(self.img_dir, img_name)).convert('RGB')
        image = self.transform(image)           # [3, img_size, img_size]
        return image, self.attrs[idx].tolist()  # [3, img_size, img_size], [40]

In [5]:
dataset = CelebA()
dataloader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=8)

print('Cantidad de imágenes:', len(dataset))
print('Cantidad de batches: ', len(dataloader))

img, attrs = dataset[0]
assert img.shape == (3, 128, 128)
assert len(attrs) == 40

In [6]:
n = 5

samples = [dataset[i] for i in random.sample(range(len(dataset)), n)]

# Imágenes:
fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.5))
for ax, (img, _) in zip(axes, samples):
    ax.imshow(img.permute(1, 2, 0))
    ax.set_axis_off()
plt.tight_layout()
plt.show()

# Atributos:
for i, (_, attrs) in enumerate(samples, 1):
    pos_attr = [name for name, val in zip(dataset.attribute_names, attrs) if val == 1]
    print(f'[{i}] {len(pos_attr)} atributos: {", ".join(pos_attr)}.')

## Modelo


In [7]:
class VAE(nn.Module):

    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
        self.img_size = 128

        self.enc = nn.Sequential(
            nn.Conv2d(3, 64, 4, stride=2, padding=1),    nn.GroupNorm(32, 64),  nn.SiLU(inplace=True),  # 3x128x128 -> 64x64x64
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  nn.GroupNorm(32, 128), nn.SiLU(inplace=True),  # 64x64x64 -> 128x32x32
            nn.Conv2d(128, 256, 4, stride=2, padding=1), nn.GroupNorm(32, 256), nn.SiLU(inplace=True),  # 128x32x32 -> 256x16x16
            nn.Conv2d(256, 512, 4, stride=2, padding=1), nn.GroupNorm(32, 512), nn.SiLU(inplace=True),  # 256x16x16 -> 512x8x8
            nn.Conv2d(512, 512, 4, stride=2, padding=1), nn.GroupNorm(32, 512), nn.SiLU(inplace=True),  # 512x8x8 -> 512x4x4
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 2 * latent_dim)
        )

        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 512 * 4 * 4),
            nn.Unflatten(1, (512, 4, 4)),
            nn.ConvTranspose2d(512, 512, 4, stride=2, padding=1), nn.GroupNorm(32, 512), nn.SiLU(inplace=True),  # 512x4x4 -> 512x8x8
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1), nn.GroupNorm(32, 256), nn.SiLU(inplace=True),  # 512x8x8 -> 256x16x16
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.GroupNorm(32, 128), nn.SiLU(inplace=True),  # 256x16x16 -> 128x32x32
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  nn.GroupNorm(32, 64),  nn.SiLU(inplace=True),  # 128x32x32 -> 64x64x64
            nn.ConvTranspose2d(64, 3, 4, stride=2, padding=1)  # 64x64x64 -> 3x128x128
        )

    def encode(self, x):
        mu, log_std = self.enc(x).chunk(2, dim=-1)  # [B, L] cada uno
        log_std = log_std.clamp(-4, 2)
        std = torch.exp(log_std)
        return mu, std

    def forward(self, x):
        mu, std = self.encode(x)
        z = mu + std * torch.randn_like(std)  # [B, L]
        logits = self.dec(z)  # [B, 3, 128, 128]
        return (mu, std), logits

In [8]:
# Ejemplo:

vae = VAE().to(DEVICE)

B, C, H, W = 16, 3, 128, 128
images = torch.randn(B, C, H, W).to(DEVICE)
(mu, std), logits = vae(images)

assert mu.shape == (B, vae.latent_dim)
assert std.shape == (B, vae.latent_dim)
assert logits.shape == images.shape

## Entrenamiento

In [9]:
def train_vae(model, dataloader, epochs):

    model.to(DEVICE).train()
    optimizer = torch.optim.Adam(model.parameters())
    loss_history = []

    try:
        for epoch in range(1, epochs + 1):
            for x, _ in tqdm(dataloader, desc=f'Época {epoch}/{epochs}'):
                x = x.to(DEVICE, non_blocking=True)
                (mu, std), logits = model(x)

                # ELBO:
                reconstruction = F.binary_cross_entropy_with_logits(logits, x, reduction='sum') / x.size(0)
                log_std = torch.log(std + 1e-8)
                prior_matching = (-0.5 * (1 + 2 * log_std - mu.pow(2) - std.pow(2))).sum(dim=1).mean()
                loss = reconstruction + prior_matching

                # Optimización:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

                loss_history.append(loss.item())

            if epoch % 10 == 0:
                torch.save({'model_state_dict': model.state_dict(), 'loss_history': loss_history}, f'vae_celeba_epoch_{epoch}.pth')

    except KeyboardInterrupt:
        print('Entrenamiento interrumpido.')
    torch.save({'model_state_dict': model.state_dict(), 'loss_history': loss_history}, 'vae_celeba.pth')

In [10]:
model = VAE(latent_dim=128)
print(f'Cantidad de parámetros: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M\n')

#train_vae(model, dataloader, epochs=80)

In [11]:
model = VAE(latent_dim=128)

checkpoint = torch.load('vae_celeba.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE).eval()

# Curva de entrenamiento:
plt.figure(figsize=(10, 4))
plt.plot(checkpoint['loss_history'], linewidth=0.5)
plt.xlabel('Iteración')
plt.ylabel('- ELBO')
plt.title('Dinámica de entrenamiento del VAE en CelebA')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Reconstrucción


In [12]:
@torch.no_grad()
def reconstruct(x, model):
    model.eval()
    (_, _), logits = model(x.to(DEVICE))
    return torch.sigmoid(logits).cpu()

n = 6
x = next(iter(dataloader))[0][:n]
x_rec = reconstruct(x, model)

fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
for i, (x_img, rec_img) in enumerate(zip(x, x_rec)):
    axes[0, i].imshow(x_img.permute(1, 2, 0)); axes[0, i].axis('off')
    axes[1, i].imshow(rec_img.permute(1, 2, 0));  axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## Aritmética en el espacio latente

In [13]:
@torch.no_grad()
def compute_all_directions(model, dataset, batch_size=256):
    model.eval().to(DEVICE)

    # mu(x) para cada imagen del dataset:
    aux_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    mus = [model.encode(x.to(DEVICE))[0] for x, _ in tqdm(aux_loader)]
    all_mus = torch.cat(mus, dim=0)  # [N, L]

    # Centroides:
    attrs = torch.from_numpy(dataset.attrs).to(DEVICE).float()  # [N, 40]
    n_pos = attrs.sum(dim=0).unsqueeze(1)                       # [40, 1]
    n_neg = attrs.shape[0] - n_pos                              # [40, 1]
    sum_pos = attrs.T @ all_mus                                 # [40, N] @ [N, L] -> [40, L]
    sum_neg = all_mus.sum(dim=0) - sum_pos                      # [40, L]
    directions = sum_pos / n_pos - sum_neg / n_neg              # [40, L]

    return {name: directions[i] for i, name in enumerate(dataset.attribute_names)}

directions = compute_all_directions(model, dataset)

In [14]:
@torch.no_grad()
def attribute_interpolation(model, image, attribute, directions=directions, lambdas=(-5, -3, -1, 1, 3, 5)):
    model.eval().to(DEVICE)
    lambdas = list(lambdas)

    # Interpolación:
    mu, _ = model.encode(image.unsqueeze(0).to(DEVICE))            # [1, L]
    lambdas_t = torch.tensor(lambdas, device=DEVICE).unsqueeze(1)  # [n_lambdas, 1]
    z = mu + lambdas_t * directions[attribute]                     # [n_lambdas, L]
    x_rec = torch.sigmoid(model.dec(z)).cpu()                      # [n_lambdas, 3, 128, 128]

    # Gráfico:
    n_neg = sum(1 for l in lambdas if l < 0)
    n_total = len(lambdas) + 1
    fig, axes = plt.subplots(1, n_total, figsize=(2 * n_total, 2.5))
    col = 0
    for lam, img in zip(lambdas, x_rec):
        if col == n_neg:
            axes[col].imshow(image.permute(1, 2, 0).numpy())
            axes[col].set_title('Original', fontsize=10, fontweight='bold')
            axes[col].axis('off')
            col += 1
        axes[col].imshow(img.permute(1, 2, 0).numpy())
        axes[col].set_title(fr'$\lambda={lam:g}$', fontsize=10)
        axes[col].axis('off')
        col += 1
    plt.suptitle(f'{attribute}', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
idx = random.sample(range(len(dataset)), 1)[0]
img, _ = dataset[idx]

for attr in dataset.attribute_names:
    attribute_interpolation(model, img, attribute=attr)